# Stage-1 exploration — results

Compare four Stage-1 seeders (`naive_pos`, `naive_abs`, `adaptive_nec`, `seqlets`) with the rest of the two-stage pipeline held fixed, on the raw ISM maps.

Run `slurm/run_explore.sbatch` first; this notebook only reads `results/*.csv` (no GPU).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RES = Path('results')
seed = pd.read_csv(RES / 'seed_comparison.csv')
summ = pd.read_csv(RES / 'explore_summary.csv')

ORDER = ['naive_pos', 'naive_abs', 'adaptive_nec', 'seqlets']
ORDER = [v for v in ORDER if v in set(seed.variant)]
COLORS = {'naive_pos': '#9467bd', 'naive_abs': '#1f77b4',
          'adaptive_nec': '#2ca02c', 'seqlets': '#ff7f0e'}
summ.set_index('variant').loc[ORDER].round(3)

In [ ]:
# 1. Recovery at each stage (median) — where does sufficiency get recovered?
stages = ['rec_naive', 'rec_stage1', 'rec_extended', 'rec_final']
labels = ['seed', 'stage1\n(necessity)', 'extended\n(2a)', 'final\n(2b)']
x = np.arange(len(stages)); w = 0.8 / len(ORDER)
fig, ax = plt.subplots(figsize=(9, 5))
for i, v in enumerate(ORDER):
    g = seed[seed.variant == v]
    ax.bar(x + i * w, [g[s].median() for s in stages], w, label=v, color=COLORS[v])
ax.axhline(0.90, ls='--', color='#888', lw=1)
ax.set_xticks(x + w * (len(ORDER) - 1) / 2); ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_ylabel('median recovery (% of WT)'); ax.set_title('Where sufficiency is recovered, by Stage-1 seeder')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 2. Core (attribution-seeded) vs context (buffered search) bp — localisation
fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(ORDER))[::-1]
core = [seed[seed.variant == v]['core_bp'].median() for v in ORDER]
flank = [seed[seed.variant == v]['flank_bp'].median() for v in ORDER]
ctx = [seed[seed.variant == v]['context_bp'].median() for v in ORDER]
ax.barh(y, core, color='#111111', label='core (necessity seed)')
ax.barh(y, flank, left=core, color='#9e9e9e', label='flank (suff. ext.)')
ax.barh(y, ctx, left=np.array(core) + np.array(flank), color='#1aa3a3', label='context (buffered)')
ax.set_yticks(y); ax.set_yticklabels(ORDER)
ax.set_xlabel('median bp'); ax.set_title('Annotation make-up: attribution-localised vs buffered')
ax.legend(); ax.grid(axis='x', alpha=0.3)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 3. Fraction of sequences with ZERO necessity-core, overall and by activity bin
bins_ = ['repressive', 'neutral', 'activating']
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(bins_)); w = 0.8 / len(ORDER)
for i, v in enumerate(ORDER):
    g = seed[seed.variant == v]
    fr = [(g[g.activity_bin == b]['core_bp'] == 0).mean() if (g.activity_bin == b).any() else np.nan for b in bins_]
    ax.bar(x + i * w, fr, w, label=v, color=COLORS[v])
ax.set_xticks(x + w * (len(ORDER) - 1) / 2); ax.set_xticklabels(bins_)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_ylabel('seqs with 0 core bp'); ax.set_title('Where the seed finds nothing (pure buffered annotation)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 4. Cost vs final recovery vs size — one marker per variant (median), IQR bars
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for v in ORDER:
    g = seed[seed.variant == v]
    for ax, xcol, xlab in ((axes[0], 'total_bp', 'median annotation length (bp)'),
                           (axes[1], 'n_experiments', 'median #model experiments')):
        mx, my = g[xcol].median(), g['rec_final'].median()
        ax.plot([g[xcol].quantile(.25), g[xcol].quantile(.75)], [my, my], color=COLORS[v], alpha=.5)
        ax.plot([mx, mx], [g['rec_final'].quantile(.25), g['rec_final'].quantile(.75)], color=COLORS[v], alpha=.5)
        ax.scatter([mx], [my], s=220, color=COLORS[v], edgecolor='white', lw=2, label=v, zorder=5)
        ax.set_xlabel(xlab)
for ax in axes:
    ax.axhline(0.90, ls='--', color='#888', lw=1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_ylabel('median final recovery'); ax.grid(alpha=0.3)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
axes[0].set_title('Recovery vs size'); axes[1].set_title('Recovery vs cost')
axes[0].legend()
plt.tight_layout(); plt.show()